# YOLOV26 — Benchmark on OOD Dataset (22 classes)


In [1]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch: 2.6.0+cu124
CUDA: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
VRAM: 8.6 GB


In [2]:
from ultralytics import YOLO
from pathlib import Path
import pandas as pd
import numpy as np
import time
import gc, torch, os

# Only evaluate saved weights for tables by default.
RUN_TRAINING = False
BENCHMARK_VARIANTS = ["yolov26n", "yolov26s", "yolov26m"]

WORKERS = 2
AMP = True
RESUME_CKPT = None
RESUME_MODEL = None

DATA_YAML    = "../data.yaml"
IMG_SIZE     = 640
EPOCHS       = 70
BATCH        = 16
DEVICE       = 0
PATIENCE     = 20
RESULTS_CSV  = "benchmark_yolov26.csv"
PERCLASS_CSV = "benchmark_yolov26_perclass.csv"

CLASS_NAMES = ["bench", "bicycle", "bus", "bus_stop", "car", "crutch", "curb", "dog", "fire_hydrant", "motorcycle", "person", "pole", "spherical_roadblock", "stairs", "stop_sign", "street_light", "traffic_light", "train", "tree", "truck", "warning_column", "waste_container"]

MODELS = [
    "yolov26n.pt",
    "yolov26s.pt",
    "yolov26m.pt",
]

# ── NEW: Enhanced Settings ──
USE_SAHI = False
USE_PREPROCESSING = False
SAHI_SLICE_SIZE = 320
SAHI_OVERLAP = 0.2


Working directory: c:\Users\admin\PFE\PFE-Obstacle-Detection\benchmark
DATA_YAML path: C:\Users\admin\PFE\PFE-Obstacle-Detection\data.yaml
✓ Found runs/detect at: C:\Users\admin\PFE\PFE-Obstacle-Detection\runs\detect


WindowsPath('C:/Users/admin/PFE/PFE-Obstacle-Detection/runs/detect')

In [3]:
# Debug: Check if weights files are found
from pathlib import Path

def _runs_detect_dir() -> Path:
    for rel in ("../runs/detect", "runs/detect"):
        p = Path(rel).resolve()
        if p.is_dir():
            return p
    return Path("../runs/detect").resolve()

root = _runs_detect_dir()
print(f"Root: {root}\n")

for variant in BENCHMARK_VARIANTS:
    folder = BENCHMARK_RUN_FOLDERS.get(variant)
    pinned = root / folder / "weights" / "best.pt"
    print(f"{variant}:")
    print(f"  Expected path: {pinned}")
    print(f"  Exists: {pinned.is_file()}\n")

Root: C:\Users\admin\PFE\PFE-Obstacle-Detection\runs\detect

yolo26n:
  Expected path: C:\Users\admin\PFE\PFE-Obstacle-Detection\runs\detect\yolo26n_ood\weights\best.pt
  Exists: True

yolo26s:
  Expected path: C:\Users\admin\PFE\PFE-Obstacle-Detection\runs\detect\yolo26s_ood\weights\best.pt
  Exists: True

yolo26m:
  Expected path: C:\Users\admin\PFE\PFE-Obstacle-Detection\runs\detect\yolo26m_ood\weights\best.pt
  Exists: True



In [4]:
import cv2
try:
    from sahi.predict import get_sliced_prediction
    from sahi import AutoDetectionModel
except:
    pass

def preprocess_image(img_path):
    img = cv2.imread(str(img_path))
    if img is None: return None
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    cl = clahe.apply(l)
    final = cv2.cvtColor(cv2.merge((cl,a,b)), cv2.COLOR_LAB2BGR)
    return final

def benchmark_model(model_name):
    print(f'\n{"="*60}\n  BENCHMARK: {model_name}\n{"="*60}')
    model = YOLO(model_name)
    model.train(data=DATA_YAML, epochs=EPOCHS, imgsz=IMG_SIZE, batch=BATCH, device=DEVICE, workers=WORKERS, name=f"{model_name.replace('.pt','')}_ood", patience=PATIENCE, save=True)
    best_path = Path(model.trainer.best)
    best_model = YOLO(str(best_path))
    metrics = best_model.val(data=DATA_YAML, split='test', device=DEVICE, imgsz=IMG_SIZE, workers=WORKERS)
    
    # Enhanced inference loop
    test_img_dir = TEST_IMG_DIR
    test_images = [p for p in test_img_dir.glob('*.*') if p.suffix.lower() in {'.jpg','.jpeg','.png'}][:200]
    sahi_model = None
    if USE_SAHI:
        device_str = f'cuda:{DEVICE}' if torch.cuda.is_available() else 'cpu'
        sahi_model = AutoDetectionModel.from_pretrained(model_type='ultralytics', model_path=str(best_path), device=device_str, confidence_threshold=0.25)
    
    tn, fp, neg_total, lats = 0, 0, 0, []
    for img_p in test_images:
        inp = preprocess_image(img_p) if USE_PREPROCESSING else str(img_p)
        t0 = time.perf_counter()
        if USE_SAHI and sahi_model:
            res = get_sliced_prediction(inp, sahi_model, slice_height=SAHI_SLICE_SIZE, slice_width=SAHI_SLICE_SIZE, overlap_height_ratio=SAHI_OVERLAP, overlap_width_ratio=SAHI_OVERLAP, verbose=0)
            cnt = len(res.object_prediction_list)
        else:
            res = best_model(inp, verbose=False)
            cnt = len(res[0].boxes) if res[0].boxes else 0
        lats.append((time.perf_counter()-t0)*1000)
        lbl_p = Path(str(img_p).replace('images', 'labels')).with_suffix('.txt')
        if not lbl_p.exists() or lbl_p.stat().st_size == 0:
            neg_total += 1
            if cnt == 0: tn += 1
            else: fp += 1
    
    row = {
        'model': model_name.replace('.pt',''),
        'mAP@0.5': round(float(metrics.box.map50), 4),
        'speed_ms/img': round(float(np.mean(lats)), 2),
        'neg_acc': round(tn/neg_total if neg_total > 0 else 1.0, 4),
        'size_MB': round(best_path.stat().st_size/1e6, 1),
    }
    del model, best_model
    gc.collect(); _safe_cuda_empty_cache()
    return row, {}


In [5]:
rows = []
all_per_class = {}

if RUN_TRAINING:
    print("RUN_TRAINING=True — training each model in MODELS (slow).\n")
    for model_name in MODELS:
        try:
            row, per_class = benchmark_model(model_name)
            rows.append(row)
            all_per_class[row["model"]] = per_class
            print(f"\n  mAP@0.5       : {row['mAP@0.5']}")
            print(f"  mAP@0.5:0.95  : {row['mAP@0.5:0.95']}")
            print(f"  Precision     : {row['precision']}")
            print(f"  Recall        : {row['recall']}")
            print(f"  F1            : {row['F1']}")
            print(f"  Speed         : {row['speed_ms/img']} ms/img")
            print(f"  Size          : {row['size_MB']} MB")
            print(f"  Params        : {row['params_M']} M")
        except Exception as e:
            print(f"  SKIPPED {model_name}: {e}")
            gc.collect()
            _safe_cuda_empty_cache()
else:
    print("RUN_TRAINING=False — loading each variant's newest best.pt (no training).\n")
    print(f"Resolved runs/detect -> {_runs_detect_dir()}\n")
    for variant in BENCHMARK_VARIANTS:
        wp = _find_newest_best_pt(variant)
        if wp is None:
            print(f"  Skip {variant}: no matching best.pt under {_runs_detect_dir()} (expect folder '{variant}' or '{variant}_*')")
            continue
        try:
            row, per_class = metrics_from_weights(wp, variant)
            rows.append(row)
            all_per_class[row["model"]] = per_class
            print(f"\n  {variant}: mAP@0.5={row['mAP@0.5']}  speed={row['speed_ms/img']} ms/img")
        except Exception as e:
            print(f"  SKIPPED {variant}: {e}")
            gc.collect()
            _safe_cuda_empty_cache()

_cols = [
    "model", "mAP@0.5", "mAP@0.5:0.95", "precision", "recall", "F1",
    "speed_ms/img", "size_MB", "params_M",
]
df = pd.DataFrame(rows, columns=_cols) if rows else pd.DataFrame(columns=_cols)
df.to_csv(RESULTS_CSV, index=False)

if all_per_class:
    df_pc = pd.DataFrame(all_per_class).T
else:
    df_pc = pd.DataFrame(columns=CLASS_NAMES)
df_pc.index.name = "model"
df_pc.to_csv(PERCLASS_CSV)

print(f"\nSaved -> {RESULTS_CSV} ({len(df)} row(s))")
print(f"Saved -> {PERCLASS_CSV}")

RUN_TRAINING=False — loading each variant's newest best.pt (no training).

Resolved runs/detect -> C:\Users\admin\PFE\PFE-Obstacle-Detection\runs\detect


====
  METRICS ONLY: yolo26n
  weights: C:\Users\admin\PFE\PFE-Obstacle-Detection\runs\detect\yolo26n_ood\weights\best.pt
====
Ultralytics 8.4.30  Python-3.11.0 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLO26n summary (fused): 122 layers, 2,379,126 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 1246.1640.9 MB/s, size: 130.0 KB)
val: Scanning C:\Users\admin\PFE\PFE-Obstacle-Detection\dataset\test\labels.cache... 1000 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1000/1000  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 63/63 8.3it/s 7.6s0.1s
                   all       1000       2945      0.664      0.572      0.611      0.436
                 bench         67        115       0.63      0.565    

In [6]:
import pandas as pd
df = pd.read_csv(RESULTS_CSV)
print(df.to_string(index=False))
styled = df.style.highlight_max(subset=['mAP@0.5', 'neg_acc'], color='#2d6a2e')
display(styled)


====
  YOLOv26 BENCHMARK — OOD Dataset (22 classes)
====

   model  mAP@0.5  mAP@0.5:0.95  precision  recall     F1  speed_ms/img  size_MB  params_M
yolo26n   0.6113        0.4361     0.6637  0.5724 0.6147         17.11      5.4       2.4
yolo26s   0.6337        0.4607     0.7147  0.5837 0.6426         19.34     20.3       9.5
yolo26m   0.6404        0.4649     0.6893  0.5998 0.6414         30.31     44.1      20.4 



model,mAP@0.5,mAP@0.5:0.95,precision,recall,F1,speed_ms/img,size_MB,params_M
yolo26n,0.6113,0.4361,0.6637,0.5724,0.6147,17.1 ms,5.4 MB,2.4 M
yolo26s,0.6337,0.4607,0.7147,0.5837,0.6426,19.3 ms,20.3 MB,9.5 M
yolo26m,0.6404,0.4649,0.6893,0.5998,0.6414,30.3 ms,44.1 MB,20.4 M
